## **Data reading**

In [0]:
df = spark.read.format("parquet")\
    .option('inferschema',True)\
    .load('abfss://bronze@carsalesdata.dfs.core.windows.net/raw_data')

In [0]:
display(df)

## **Data Transformation**

In [0]:
from pyspark.sql import functions as F

In [0]:
df = df.withColumn('Model_Category',F.split(df['Model_ID'],'-')[0])

In [0]:
df.display()

In [0]:
df = df.withColumn('revenue_per_unit',df['Revenue']/df['Units_Sold'])
df.display()

# **AD-HOC Analysis**
## Data Aggregation
How many units were sold of each branch every year. To know which branch is doing good and which is doing bad?

In [0]:
df.groupBy('Year','BranchName').agg(
    F.sum('Units_sold').alias('Total_Units_Sold')
).sort('Year','Total_Units_Sold',ascending=[True,False]).display()

# **Data Writing**

In [0]:
df.write.format('parquet')\
    .mode('append')\
        .option('path','abfss://silver@carsalesdata.dfs.core.windows.net/carsales')\
            .save()
           


# querying silver data

In [0]:
%sql
select * from parquet.`abfss://silver@carsalesdata.dfs.core.windows.net/carsales`;